# Attack Transition Analysis — Test1 & Test2
Shows sensor behaviour in a **200-second window** around each attack onset (100 s before → 100 s after).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

LABEL_NAMES = {
    0: 'Normal',
    1: 'AP Simple',
    2: 'AP Combined',
    3: 'AE Simple',
    4: 'AE Combined',
}
LABEL_COLORS = {
    0: '#2ecc71',
    1: '#e74c3c',
    2: '#c0392b',
    3: '#3498db',
    4: '#1a5276',
}
WINDOW_SEC = 100   # seconds before and after attack start
BASE_PATH  = '../data/processed'

In [ ]:
def load_dataset(name):
    """Load a processed split from data/processed/."""
    path = f'{BASE_PATH}/{name}.csv'
    df = pd.read_csv(path, low_memory=False)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    return df

def get_sensor_cols(df):
    meta = {'timestamp','attack_id','scenario','attack_type',
            'combination','target_controller','target_points','label','attack'}
    return [c for c in df.columns if c not in meta]

def get_attack_blocks(df):
    """Return list of (start_ts, end_ts, label) for each contiguous attack block."""
    df = df.copy()
    df['_grp'] = (df['label'] != df['label'].shift()).cumsum()
    blocks = []
    for _, grp in df[df['label'] > 0].groupby('_grp'):
        blocks.append({
            'start': grp['timestamp'].iloc[0],
            'end':   grp['timestamp'].iloc[-1],
            'label': int(grp['label'].iloc[0]),
            'target': grp['target_points'].iloc[0] if 'target_points' in grp.columns else '',
        })
    return blocks

def top_affected_sensors(df, block, sensor_cols, n=4):
    """Pick n sensors with the biggest mean shift inside vs outside the attack window."""
    start, end = block['start'], block['end']
    mask_atk = (df['timestamp'] >= start) & (df['timestamp'] <= end)
    mask_norm = ~mask_atk
    if mask_atk.sum() < 2 or mask_norm.sum() < 2:
        return sensor_cols[:n]
    diff = (df.loc[mask_atk, sensor_cols].mean() -
            df.loc[mask_norm, sensor_cols].mean()).abs()
    return diff.nlargest(n).index.tolist()

def plot_attack_window(df, block, sensor_cols, ax_sensors, ax_label, title=''):
    """Plot sensor traces + label band for one 200-sec window."""
    start = block['start']
    t0 = start - pd.Timedelta(seconds=WINDOW_SEC)
    t1 = start + pd.Timedelta(seconds=WINDOW_SEC)
    win = df[(df['timestamp'] >= t0) & (df['timestamp'] <= t1)].copy()
    if win.empty:
        return
    win['rel'] = (win['timestamp'] - start).dt.total_seconds()

    sensors = top_affected_sensors(df, block, sensor_cols, n=4)
    colors  = ['#2c3e50', '#8e44ad', '#e67e22', '#16a085']

    for i, col in enumerate(sensors):
        vals = win[col].values.astype(float)
        vmin, vmax = np.nanmin(vals), np.nanmax(vals)
        normed = (vals - vmin) / (vmax - vmin + 1e-9)
        ax_sensors.plot(win['rel'], normed, color=colors[i], linewidth=1.5, label=col)

    ax_sensors.axvline(0, color='red', linestyle='--', linewidth=1.2, label='Attack start')
    ax_sensors.set_xlim(-WINDOW_SEC, WINDOW_SEC)
    ax_sensors.set_ylabel('Normalised value')
    ax_sensors.set_title(title, fontsize=9, fontweight='bold')
    ax_sensors.legend(fontsize=7, loc='upper left', ncol=2)

    for _, row in win.iterrows():
        ax_label.axvspan(row['rel'] - 0.5, row['rel'] + 0.5,
                         color=LABEL_COLORS.get(row['label'], 'grey'), alpha=0.8)
    ax_label.set_xlim(-WINDOW_SEC, WINDOW_SEC)
    ax_label.set_yticks([])
    ax_label.set_xlabel('Seconds relative to attack start')
    ax_label.set_ylabel('Label', fontsize=7)

    present_labels = win['label'].unique()
    patches = [mpatches.Patch(color=LABEL_COLORS[l], label=LABEL_NAMES[l])
               for l in sorted(present_labels)]
    ax_label.legend(handles=patches, fontsize=7, loc='upper left', ncol=3)

## Test1 — AP Simple attacks

In [ ]:
df1 = load_dataset('test1')
sensor_cols1 = get_sensor_cols(df1)
blocks1 = get_attack_blocks(df1)

print(f'Test1: {len(df1):,} rows | {len(blocks1)} attack blocks')
for b in blocks1:
    print(f'  [{LABEL_NAMES[b["label"]]}]  {b["start"]}  target={b["target"]}')

In [ ]:
# Show first 6 attacks side-by-side (2 per row)
blocks_to_show = blocks1[:6]
n_cols = 2
n_rows = len(blocks_to_show)

fig, axes = plt.subplots(
    nrows=n_rows, ncols=n_cols,
    figsize=(14, n_rows * 2.8),
    gridspec_kw={'height_ratios': [1] * n_rows, 'hspace': 0.55, 'wspace': 0.3}
)
# Re-structure: each attack needs 2 sub-rows (sensors + label)
fig2, axes2 = plt.subplots(
    nrows=n_rows * 2, ncols=1,
    figsize=(13, n_rows * 3),
    gridspec_kw={
        'height_ratios': [4, 1] * n_rows,
        'hspace': 0.6
    }
)
plt.close(fig)  # close unused

for i, block in enumerate(blocks_to_show):
    ax_s = axes2[i * 2]
    ax_l = axes2[i * 2 + 1]
    title = (f'Attack {i+1}  |  {LABEL_NAMES[block["label"]]}  '
             f'|  {block["start"].strftime("%H:%M:%S")}  '
             f'|  target: {block["target"]}')
    plot_attack_window(df1, block, sensor_cols1, ax_s, ax_l, title=title)

fig2.suptitle('Test1 — Sensor transitions around attack onset (±100 s)', fontsize=12, y=1.01)
plt.savefig('../outputs/test1_attack_transitions.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# Remaining attacks (7-14)
blocks_rest = blocks1[6:]
if blocks_rest:
    n_rows2 = len(blocks_rest)
    fig3, axes3 = plt.subplots(
        nrows=n_rows2 * 2, ncols=1,
        figsize=(13, n_rows2 * 3),
        gridspec_kw={'height_ratios': [4, 1] * n_rows2, 'hspace': 0.6}
    )
    if n_rows2 == 1:
        axes3 = list(axes3)
    for i, block in enumerate(blocks_rest):
        ax_s = axes3[i * 2]
        ax_l = axes3[i * 2 + 1]
        title = (f'Attack {i+7}  |  {LABEL_NAMES[block["label"]]}  '
                 f'|  {block["start"].strftime("%H:%M:%S")}  '
                 f'|  target: {block["target"]}')
        plot_attack_window(df1, block, sensor_cols1, ax_s, ax_l, title=title)
    fig3.suptitle('Test1 — Attacks 7-14', fontsize=12, y=1.01)
    plt.savefig('../outputs/test1_attack_transitions_2.png', bbox_inches='tight', dpi=130)
    plt.show()

## Test2 — AP Simple, AP Combined, AE Simple

In [ ]:
df2 = load_dataset('test2')
sensor_cols2 = get_sensor_cols(df2)
blocks2 = get_attack_blocks(df2)

print(f'Test2: {len(df2):,} rows | {len(blocks2)} attack blocks')
label_summary = {}
for b in blocks2:
    label_summary[b['label']] = label_summary.get(b['label'], 0) + 1
print('By type:', {LABEL_NAMES[k]: v for k, v in label_summary.items()})

In [ ]:
# Plot one representative attack per label type
label_types = [1, 2, 3]   # AP simple, AP combined, AE simple
representative = {}
for b in blocks2:
    l = b['label']
    if l not in representative:
        representative[l] = b

n_types = len(representative)
fig4, axes4 = plt.subplots(
    nrows=n_types * 2, ncols=1,
    figsize=(13, n_types * 3.2),
    gridspec_kw={'height_ratios': [4, 1] * n_types, 'hspace': 0.6}
)

for i, (lbl, block) in enumerate(sorted(representative.items())):
    ax_s = axes4[i * 2]
    ax_l = axes4[i * 2 + 1]
    title = (f'{LABEL_NAMES[lbl]}  |  {block["start"].strftime("%Y-%m-%d %H:%M:%S")}  '
             f'|  target: {block["target"]}')
    plot_attack_window(df2, block, sensor_cols2, ax_s, ax_l, title=title)

fig4.suptitle('Test2 — One representative attack per type (±100 s)', fontsize=12, y=1.01)
plt.savefig('../outputs/test2_attack_by_type.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# All test2 attacks — paginated 8 per figure
PAGE = 8
for page_idx in range(0, len(blocks2), PAGE):
    page_blocks = blocks2[page_idx: page_idx + PAGE]
    n_r = len(page_blocks)
    fig5, axes5 = plt.subplots(
        nrows=n_r * 2, ncols=1,
        figsize=(13, n_r * 3),
        gridspec_kw={'height_ratios': [4, 1] * n_r, 'hspace': 0.6}
    )
    for i, block in enumerate(page_blocks):
        ax_s = axes5[i * 2]
        ax_l = axes5[i * 2 + 1]
        title = (f'Attack {page_idx+i+1}  |  {LABEL_NAMES[block["label"]]}  '
                 f'|  {block["start"].strftime("%m-%d %H:%M:%S")}  '
                 f'|  target: {block["target"]}')
        plot_attack_window(df2, block, sensor_cols2, ax_s, ax_l, title=title)
    page_num = page_idx // PAGE + 1
    fig5.suptitle(f'Test2 — All attacks (page {page_num})', fontsize=12, y=1.01)
    plt.savefig(f'../outputs/test2_all_attacks_p{page_num}.png', bbox_inches='tight', dpi=130)
    plt.show()

## Summary — Label distribution over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

for ax, (df, name) in zip(axes, [(df1, 'Test1'), (df2, 'Test2')]):
    for lbl in sorted(df['label'].unique()):
        mask = df['label'] == lbl
        ax.scatter(
            df.loc[mask, 'timestamp'], 
            [lbl] * mask.sum(),
            c=LABEL_COLORS[lbl], s=1, label=LABEL_NAMES[lbl], rasterized=True
        )
    ax.set_yticks(list(LABEL_NAMES.keys()))
    ax.set_yticklabels(list(LABEL_NAMES.values()), fontsize=8)
    ax.set_title(f'{name} — label timeline', fontweight='bold')
    ax.legend(fontsize=7, loc='upper right', markerscale=5)

plt.tight_layout()
plt.savefig('../outputs/label_timeline.png', bbox_inches='tight', dpi=130)
plt.show()